# 27 - Scale-safe conv: does a zero-init gate make conv harmless at small scale?

Our finding: a depthwise conv HURTS a 3M model (+20% on WT103 BPE) but helps a 30M model (-10%).
The twist (ReZero-style): add a per-channel gate to the conv, initialized at ZERO, so the conv starts
as a no-op and the model opens it only if it helps. Decisive first test at 3M:

* `combo`              - reference
* `combo_conv3`        - plain conv (expected +20% HARM)
* `combo_conv3_gated`  - zero-init gated conv (does it return to ~combo, i.e. harm removed?)

If gated ~= combo, the gate makes conv safe to include at small scale (then we test 30M keeps the gain).
If gated still ~= +20%, or it kills the model, the twist fails. 3M, WT103 BPE, block256 batch16, 2 seeds.


In [1]:
%matplotlib inline
import os, math, time, csv, json, gc
from contextlib import nullcontext
import numpy as np
import torch
from model import Config, GPT, count_params

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
D = "../dataset"
train_data = torch.from_numpy(np.load(D + "/wt103_bpe_train.npy"))
val_data = torch.from_numpy(np.load(D + "/wt103_bpe_val.npy"))
vocab = json.load(open(D + "/wt103_bpe_meta.json"))["vocab"]
print(f"vocab={vocab}  train={len(train_data):,}  val={len(val_data):,}")

BLOCK, BATCH = 256, 16
def get_batch(split, gen):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK, (BATCH,), generator=gen)
    x = torch.stack([d[i:i + BLOCK] for i in ix]); y = torch.stack([d[i + 1:i + 1 + BLOCK] for i in ix])
    return x.to(device), y.to(device)
def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()

def train_one(over, seed=0, max_iters=3000, eval_iters=200, lr=3e-4, warmup=150, min_lr=3e-5):
    torch.manual_seed(seed)
    cfg = Config(vocab_size=vocab, block_size=BLOCK, n_layer=4, n_head=4, n_embd=256, dropout=0.1, **over)
    model = GPT(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=0.1)
    tg = torch.Generator().manual_seed(7777 + seed)
    def get_lr(it):
        if it < warmup: return lr * (it + 1) / warmup
        r = (it - warmup) / max(1, max_iters - warmup)
        return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * r))
    @torch.no_grad()
    def ev():
        model.eval(); g = torch.Generator().manual_seed(4242); tot = 0.0
        for _ in range(eval_iters):
            x, y = get_batch("val", g)
            with autocast_ctx():
                _, loss = model(x, y)
            tot += loss.item()
        model.train(); return tot / eval_iters
    for it in range(max_iters):
        for grp in opt.param_groups: grp["lr"] = get_lr(it)
        x, y = get_batch("train", tg)
        with autocast_ctx():
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    ppl = math.exp(ev()); pm = count_params(model) / 1e6
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return ppl, pm
print("ready")


device: cuda
vocab=8192  train=14,923,628  val=281,169
ready


In [2]:
COMBO = {"pos_encoding": "rope", "activation": "swiglu", "norm_placement": "pre", "norm_type": "rmsnorm"}
CONFIGS = {
    "combo":              COMBO,
    "combo_conv3":        {**COMBO, "conv_kernel": 3},
    "combo_conv3_gated":  {**COMBO, "conv_kernel": 3, "conv_gate": True},
}
SEEDS = [0, 1]
runs_csv = "../results/scalesafe_runs.csv"
done = set()
if os.path.exists(runs_csv):
    with open(runs_csv) as f:
        r = csv.reader(f); next(r, None)
        for row in r:
            if len(row) == 4: done.add((row[0], row[1]))
else:
    with open(runs_csv, "w", newline="") as f:
        csv.writer(f).writerow(["config", "seed", "params_M", "val_ppl"])
t0 = time.time()
for name, over in CONFIGS.items():
    for s in SEEDS:
        if (name, str(s)) in done: continue
        p, pm = train_one(over, seed=s)
        with open(runs_csv, "a", newline="") as f:
            csv.writer(f).writerow([name, s, f"{pm:.3f}", f"{p:.4f}"])
        print(f"  {name:18s} seed {s}: ppl {p:.3f} ({pm:.2f}M) | {(time.time()-t0)/60:.1f} min", flush=True)
print(f"ALL DONE in {(time.time()-t0)/60:.1f} min")


  combo              seed 0: ppl 126.600 (5.25M) | 1.7 min


  combo              seed 1: ppl 126.163 (5.25M) | 3.5 min


  combo_conv3        seed 0: ppl 150.095 (5.26M) | 5.5 min


  combo_conv3        seed 1: ppl 153.593 (5.26M) | 7.4 min


  combo_conv3_gated  seed 0: ppl 125.087 (5.26M) | 9.5 min


  combo_conv3_gated  seed 1: ppl 125.911 (5.26M) | 11.5 min


ALL DONE in 11.5 min


In [3]:
import numpy as np, csv
rows = {}
with open("../results/scalesafe_runs.csv") as f:
    r = csv.reader(f); next(r, None)
    for c, s, pm, pp in r: rows.setdefault(c, []).append(float(pp))
m = {k: float(np.mean(v)) for k, v in rows.items()}
co = m["combo"]
print(f"3M, WT103 BPE (2 seeds):")
for c in ["combo", "combo_conv3", "combo_conv3_gated"]:
    print(f"  {c:18s} {m[c]:.3f}  ({100*(m[c]-co)/co:+.2f}% vs combo)")
harm = 100*(m["combo_conv3"]-co)/co
gated = 100*(m["combo_conv3_gated"]-co)/co
print(f"\nplain conv harm: {harm:+.1f}% ; gated conv: {gated:+.1f}%")
if gated <= 1.0 and harm > 3:
    print("=> SUCCESS: the zero-init gate REMOVES the small-scale harm (conv now safe at 3M).")
elif gated < harm - 3:
    print("=> PARTIAL: gate reduces but does not fully remove the harm.")
else:
    print("=> FAILS: gate does not rescue the small-scale harm.")


3M, WT103 BPE (2 seeds):
  combo              126.381  (+0.00% vs combo)
  combo_conv3        151.844  (+20.15% vs combo)
  combo_conv3_gated  125.499  (-0.70% vs combo)

plain conv harm: +20.1% ; gated conv: -0.7%
=> SUCCESS: the zero-init gate REMOVES the small-scale harm (conv now safe at 3M).


## Reading this

Decisive go/no-go for the scale-safe twist. SUCCESS (gated ~= combo) means the zero-init gate makes a
scale-dependent module harmless at small scale - then the next test is whether it KEEPS the -10% gain
at 30M (giving a module that is safe to include at any scale). Honest negative is also informative.
